In [ ]:
import pandas as pd

In [ ]:
data= pd.read_csv('IMDB_Dataset.csv')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

data['sentiment']=le.fit_transform(data['sentiment'])
data['sentiment']

In [ ]:
data.columns

In [ ]:
x= data['review'].values
y= data['sentiment'].values

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
vocab_siz= 10000
Max_len = 200 

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer= Tokenizer(num_words=vocab_siz, oov_token='<OOV>')
tokenizer.fit_on_texts(x_train)
x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_train_pad = pad_sequences(x_train_seq,maxlen=Max_len,padding='post',truncating='post')
x_test_pad = pad_sequences(x_test_seq,maxlen=Max_len,padding='post',truncating='post')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,Flatten

model = Sequential()

model.add(Embedding(input_dim=vocab_siz,output_dim=64,input_length=Max_len))
model.add(Flatten())
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
h= model.fit(
    x_train_pad,y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


In [ ]:
loss, accuracy =model.evaluate(x_test_pad,y_test,verbose=1)
print("Accuracy : ",accuracy)
print("Test Loss: ",loss)

In [ ]:
y_pred_prob= model.predict(x_test_pad)
y_pred= (y_pred_prob>0.5).astype(int)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negative', 'positive'],
            yticklabels=['negative', 'positive'])
plt.xlabel('predicted')
plt.ylabel('actual')
plt.title('confusion matrix')
plt.show()

In [ ]:
print(cm)

In [ ]:

# classification report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['negative', 'positive']))